# Lab 14 · Thực hành kể chuyện: kim tự tháp, lời đúng mức và thẩm định

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 14**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook bài giảng Bài 14 thẩm định một báo cáo do AI tạo, gồm năm kết luận. Trong bài thực hành,
bạn sẽ tự viết kết luận đúng mức, tạo một ví dụ về nghịch lý Simpson và thẩm định một kết luận mới.
Tuần này cũng là **tuần nộp bài tập lớn**; 30 phút cuối dành để kiểm tra trước khi nộp.

**Mục tiêu bài lab**

1. Sắp xếp một mục báo cáo theo cấu trúc kim tự tháp (kết luận → bằng chứng → giới hạn).
2. Phân biệt cách diễn đạt đúng mức và quá mức trên các cặp số liệu–kết luận cụ thể.
3. Tự tính một nghịch lý Simpson: hai nhóm cùng tăng, tổng gộp vẫn giảm.
4. Thẩm định một kết luận mới bằng quy trình bốn bước.

Phiên bản đề: `storytelling-review-v1` (25/09/2026). Lưu đúng tên **`lab-14.ipynb`**.

## Cách làm và nộp bài

- Với phần có hướng dẫn, hãy **tự gõ, không dùng AI**. Đây là chế độ 🚫 đóng, tương ứng với các bài kiểm tra định kỳ ở giờ lý thuyết.
  Bài tự làm cuối lab ở chế độ ✅ mở: được dùng AI nhưng phải khai báo theo chính sách của môn học.
- Bài 1, 2, 6 là **câu trả lời gán vào biến** (`thu_tu`, `nhan`, `phan_quyet`) — grader đọc giá trị biến.
  Public check chỉ kiểm **định dạng**, không cho biết đúng/sai; tự lập luận trước khi xem phần diễn giải của giảng viên.
- Bài 3, 4, 5 là **hàm**: grader chạy riêng cell đó với bảng khác (tên cột, số kỳ khác…), không phụ thuộc biến `bang`, `ds` ở cell khác.
  `pd`, `np` được cung cấp sẵn. Giữ tên hàm/biến, tham số, cell ID và metadata; không ghi cứng kết quả.
- Một check lỗi không dừng các check sau. TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa.
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Bài 1 kim tự tháp 10 · Bài 2 năm cặp đúng mức/quá mức 15 (3 điểm mỗi cặp) | 25 | Tự động |
| Bài 3 giá trung bình gộp 20 · Bài 4 tỷ trọng phân khúc 10 | 30 | Tự động |
| Bài 5 truy số 15 · Bài 6 phán quyết thẩm định 10 | 25 | Tự động |
| Lập luận cho Bài 2, nguyên nhân nghịch lý Simpson, lý do phán quyết | 15 | Giảng viên |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Phần 🧭 hỗ trợ bài tập lớn ở cuối lab làm theo nhóm, không tính vào điểm lab.

In [1]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")

In [2]:
import numpy as np
import pandas as pd

## Bài 1 · Xếp kim tự tháp

Bốn câu dưới đây thuộc một mục báo cáo về giá tại Santiago nhưng đang bị xáo trộn.
Xếp lại theo **kim tự tháp**: (1) kết luận → (2) bằng chứng → (3) giới hạn →
(4) chi tiết phương pháp (đặt ở cuối hoặc trong phụ lục).

### Câu trả lời · 10 điểm tự động

Gán biến `thu_tu` là **list 4 chữ cái** (mỗi chữ trong `CAU` đúng một lần) theo thứ tự kim tự tháp.

In [3]:
CAU = {
 "A": "Con số dựa trên 17.688 chỗ ở có giá hợp lệ ở mốc chụp 29/06/2026; 846 chỗ ở không có giá đã được loại và ghi trong qa_report.",
 "B": "Giá thuê điển hình ở Santiago là 59.000 CLP/đêm (khoảng 1,65 triệu đồng), thấp hơn mức trung bình 118.000 CLP.",
 "C": "Trung vị ít bị ảnh hưởng bởi giá ngoại lai: 177 giá trị cực đoan đã gắn cờ làm trung bình cao hơn 44% nhưng gần như không làm thay đổi trung vị.",
 "D": "Kết luận chỉ áp dụng cho giá niêm yết một đêm, chưa gồm phí vệ sinh/dịch vụ; giá thực tế khách trả có thể cao hơn.",
}
for k, v in CAU.items():
    print(f"{k}. {v[:100]}…")

A. Con số dựa trên 17.688 chỗ ở có giá hợp lệ ở mốc chụp 29/06/2026; 846 chỗ ở không có giá đã được loạ…
B. Giá thuê điển hình ở Santiago là 59.000 CLP/đêm (khoảng 1,65 triệu đồng), thấp hơn mức trung bình 11…
C. Trung vị ít bị ảnh hưởng bởi giá ngoại lai: 177 giá trị cực đoan đã gắn cờ làm trung bình cao hơn 44…
D. Kết luận chỉ áp dụng cho giá niêm yết một đêm, chưa gồm phí vệ sinh/dịch vụ; giá thực tế khách trả c…


In [4]:
thu_tu = ["B", "C", "D", "A"]


In [5]:
def check_q1():
    if ... in thu_tu:
        raise NotImplementedError
    assert isinstance(thu_tu, list) and sorted(thu_tu) == sorted(CAU), "Mỗi chữ A–D xuất hiện đúng một lần"
    print("Định dạng hợp lệ — đáp án do grader chấm.")

public_check("Q1 · định dạng", check_q1)

Định dạng hợp lệ — đáp án do grader chấm.
[ĐẠT] Q1 · định dạng: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


**Lý do sắp xếp:** B → C → D → A.

- **B — kết luận:** đưa ngay mức giá điển hình và đặt cạnh giá trung bình để người đọc nắm thông điệp chính.
- **C — bằng chứng:** nêu ảnh hưởng của giá ngoại lai lên trung bình và trung vị, giải thích vì sao chọn trung vị.
- **D — giới hạn:** chỉ nói về giá niêm yết một đêm, chưa gồm các khoản phí; không đồng nhất với tổng tiền khách trả.
- **A — phương pháp:** ghi mốc chụp, số dòng có giá hợp lệ và cách xử lý giá thiếu để người đọc truy lại phép tính.

Các số liệu và quy đổi tiền ở bốn câu được dùng theo đề; phần này chỉ sắp xếp lập luận, chưa xác minh lại snapshot hay tỷ giá.

## Bài 2 · Lời đúng mức với số liệu

Với mỗi cặp (số liệu → lời viết), gán `"vua"` (đúng mức) hoặc `"qua"` (quá mức).

### Câu trả lời · 15 điểm tự động (3 điểm mỗi cặp)

Gán biến `nhan` là dict `{1: ..., 2: ..., 3: ..., 4: ..., 5: ...}`, mỗi giá trị là `"vua"` hoặc `"qua"`.

In [6]:
CAP = {
 1: ("So cùng kỳ: +53,7% (T5/2026 so với T5/2025)", "Thị trường tăng trưởng mạnh so với cùng kỳ"),
 2: ("+3,0% so với tháng trước", "Bứt phá thần tốc trong tháng 5"),
 3: ("Hệ số tương quan 0,081", "Nhiệt độ ngày ảnh hưởng rõ rệt tới lượng đánh giá"),
 4: ("Đúng 1 chỗ ở đòi ở tối thiểu 730 đêm", "Một vài ca cực đoan cần gắn cờ"),
 5: ("Nhóm es chiếm 65,4% (quy tắc chưa kiểm định)", "Chắc chắn 2/3 khách nói tiếng Tây Ban Nha"),
}
for k, (so_lieu, loi) in CAP.items():
    print(f"{k}. {so_lieu}  →  \"{loi}\"")

1. So cùng kỳ: +53,7% (T5/2026 so với T5/2025)  →  "Thị trường tăng trưởng mạnh so với cùng kỳ"
2. +3,0% so với tháng trước  →  "Bứt phá thần tốc trong tháng 5"
3. Hệ số tương quan 0,081  →  "Nhiệt độ ngày ảnh hưởng rõ rệt tới lượng đánh giá"
4. Đúng 1 chỗ ở đòi ở tối thiểu 730 đêm  →  "Một vài ca cực đoan cần gắn cờ"
5. Nhóm es chiếm 65,4% (quy tắc chưa kiểm định)  →  "Chắc chắn 2/3 khách nói tiếng Tây Ban Nha"


In [7]:
nhan = {1: "vua", 2: "qua", 3: "qua", 4: "qua", 5: "qua"}


In [8]:
def check_q2():
    if ... in nhan.values():
        raise NotImplementedError
    assert set(nhan) == set(CAP), "Cần đúng 5 khóa 1–5"
    assert set(nhan.values()) <= {"vua", "qua"}, "Mỗi nhãn là 'vua' hoặc 'qua'"
    print("Định dạng hợp lệ — đáp án do grader chấm.")

public_check("Q2 · định dạng", check_q2)

Định dạng hợp lệ — đáp án do grader chấm.
[ĐẠT] Q2 · định dạng: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


**Lập luận và cách viết lại:**

1. **Đúng mức (`vua`):** tăng 53,7% so với cùng kỳ có thể mô tả là tăng mạnh trong ngữ cảnh bài. Khi viết báo cáo, cần gọi đúng chỉ tiêu thay cho từ chung “thị trường”: “Chỉ tiêu đang xét trong tháng 5/2026 tăng 53,7% so với tháng 5/2025”. Nếu đây là số đánh giá, phải viết “số đánh giá”, không suy thành doanh thu hoặc lượt đặt phòng.
2. **Quá mức (`qua`):** “bứt phá thần tốc” phóng đại mức tăng 3,0%. Viết: “Chỉ tiêu tăng 3,0% so với tháng trước”; chưa đủ dữ liệu để kết luận về xu hướng dài hạn hay mùa vụ.
3. **Quá mức (`qua`):** “ảnh hưởng rõ rệt” vừa cường điệu hệ số tương quan 0,081, vừa gợi quan hệ nhân quả. Viết: “Trong mẫu đang xét, nhiệt độ ngày và lượng đánh giá có tương quan tuyến tính dương rất yếu (r = 0,081); kết quả này không chứng minh quan hệ nhân quả”. Tương quan nhỏ cũng không loại trừ một quan hệ phi tuyến.
4. **Quá mức (`qua`):** “một vài ca” không khớp với đúng một chỗ ở được phát hiện. Viết: “Có một chỗ ở yêu cầu thời gian lưu trú tối thiểu 730 đêm, cần gắn cờ để kiểm tra”. Giá trị cực đoan chưa chắc là lỗi, không nên tự động xóa.
5. **Quá mức (`qua`):** “chắc chắn” vượt quá độ tin cậy của quy tắc chưa kiểm định; nhãn ngôn ngữ bình luận không đồng nghĩa ngôn ngữ của mọi khách. Viết: “Quy tắc hiện tại gán nhãn es cho 65,4% đơn vị văn bản được phân loại; cần đối chiếu nhãn tay trước khi kết luận về ngôn ngữ”. Cần nêu mẫu số và phạm vi mẫu, không suy tỷ lệ văn bản thành tỷ lệ khách.

## Bài 3 · Tự tạo một nghịch lý Simpson

Hai phân khúc **cùng tăng giá 10%**, nhưng giá trung bình gộp lại giảm. Hãy tính để kiểm tra.
Các số dưới đây chỉ dùng để minh hoạ cơ chế, không phải dữ liệu Santiago.

### Hợp đồng hàm · 20 điểm tự động

`pooled_means(bang: pd.DataFrame, ky: str = "ky", n: str = "n", gia: str = "gia_tb") -> pd.Series`

Nhận bảng mỗi dòng là một (kỳ, phân khúc) với số chỗ ở ở cột `n` và giá trung bình của phân khúc ở cột `gia`
(tên cột truyền qua tham số). Trả Series: index là các kỳ, giá trị là **giá trung bình gộp** của kỳ đó
`tổng(n × gia) / tổng(n)` (trung bình có trọng số, **không** phải trung bình cộng các dòng). Không sửa input, không làm tròn.

In [9]:
bang = pd.DataFrame({
    "ky":        ["T9",  "T9",  "T6",  "T6"],
    "phan_khuc": ["cao", "re",  "cao", "re"],
    "n":         [100,   100,   80,    220],
    "gia_tb":    [60.0,  30.0,  66.0,  33.0],   # nghìn CLP — mỗi phân khúc +10%
})
bang

,ky,phan_khuc,n,gia_tb
0,T9,cao,100,60.0
1,T9,re,100,30.0
2,T6,cao,80,66.0
3,T6,re,220,33.0


In [10]:
def pooled_means(bang: pd.DataFrame, ky: str = "ky", n: str = "n", gia: str = "gia_tb") -> pd.Series:
    tong_gia_co_trong_so = (bang[n] * bang[gia]).groupby(bang[ky], observed=True).sum()
    tong_so_cho_o = bang.groupby(ky, observed=True)[n].sum()
    return tong_gia_co_trong_so / tong_so_cho_o


In [11]:
def check_q3():
    gop = pooled_means(bang)
    assert isinstance(gop, pd.Series) and set(gop.index) == {"T6", "T9"}
    assert round(gop["T9"], 1) == 45.0 and round(gop["T6"], 1) == 41.8, "gộp: 45.0 → 41.8"
    assert gop["T6"] != bang.loc[bang["ky"] == "T6", "gia_tb"].mean(), "Cần trung bình có trọng số theo n"

public_check("Q3 · ví dụ công khai", check_q3)

[ĐẠT] Q3 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


## Bài 4 · Cơ cấu mẫu thay đổi

### Hợp đồng hàm · 10 điểm tự động

`segment_share(bang: pd.DataFrame, phan_khuc: str, ky: str = "ky", nhom: str = "phan_khuc", n: str = "n") -> pd.Series`

Nhận bảng như Bài 3 và tên một phân khúc. Trả Series: index là các kỳ, giá trị là **tỷ trọng số chỗ ở** của phân khúc đó
trong kỳ (tổng `n` của phân khúc / tổng `n` của kỳ, thang 0–1); kỳ không có phân khúc đó thì tỷ trọng là 0. Không sửa input.

In [12]:
def segment_share(bang: pd.DataFrame, phan_khuc: str, ky: str = "ky", nhom: str = "phan_khuc", n: str = "n") -> pd.Series:
    tong_so_cho_o = bang.groupby(ky, observed=True)[n].sum()
    so_cho_o_nhom = (
        bang.loc[bang[nhom].eq(phan_khuc)]
        .groupby(ky, observed=True)[n].sum()
        .reindex(tong_so_cho_o.index, fill_value=0)
    )
    return so_cho_o_nhom / tong_so_cho_o


In [13]:
def check_q4():
    ty_trong = segment_share(bang, "re")
    assert isinstance(ty_trong, pd.Series) and set(ty_trong.index) == {"T6", "T9"}
    assert ty_trong["T9"] == 0.5 and round(ty_trong["T6"], 2) == 0.73
    assert segment_share(bang[bang["phan_khuc"] == "cao"], "re").tolist() == [0.0, 0.0]

public_check("Q4 · ví dụ công khai", check_q4)

[ĐẠT] Q4 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [14]:
def simpson():
    gop, re = pooled_means(bang), segment_share(bang, "re")
    return (f"Mỗi phân khúc +10%, nhưng gộp: {gop['T9']:.1f} → {gop['T6']:.1f} (GIẢM {(1 - gop['T6'] / gop['T9']):.0%}).\n"
            f"Tỷ trọng phân khúc rẻ: {re['T9']:.0%} → {re['T6']:.0%}.")

preview("Bài 3–4", simpson)

Bài 3–4
Mỗi phân khúc +10%, nhưng gộp: 45.0 → 41.8 (GIẢM 7%).
Tỷ trọng phân khúc rẻ: 50% → 73%.


**Nguyên nhân — dữ liệu minh họa, đơn vị nghìn CLP:**

- Kỳ T9: (100 × 60 + 100 × 30) / (100 + 100) = **45,0**.
- Kỳ T6: (80 × 66 + 220 × 33) / (80 + 220) = **41,8**.
- Giá mỗi phân khúc tăng 10%, nhưng tỷ trọng phân khúc rẻ tăng từ 100/200 = **50%** lên 220/300 ≈ **73,33%**. Nhóm giá thấp chiếm nhiều trọng số hơn, kéo giá gộp giảm (41,8/45 − 1) × 100 ≈ **−7,11%**.
- Nếu giữ trọng số T9 (50% mỗi nhóm), giá T6 sẽ là (66 + 33)/2 = **49,5**, tăng 10%. Con số này tách tác động giá trong nhóm khỏi thay đổi cơ cấu mẫu.

Khi so hai mốc chụp, cần kiểm tra tỷ trọng phân khúc, quận và loại phòng; số listing vào/ra mẫu; dữ liệu thiếu và tiêu chí lọc; cùng đơn vị giá, phạm vi địa lý và định nghĩa chỉ tiêu. Nên báo cáo cả số gộp lẫn từng nhóm, và có thể so trên tập listing chung hoặc trọng số cố định. Giá gộp giảm không chứng minh mọi chỗ ở đều giảm giá.

## Bài 5 · Thẩm định một kết luận mới — bước 1: truy số

Kết luận này chưa có trong notebook bài giảng: *"Chủ nhà chuyên nghiệp **được khách yêu thích hơn**
so với chủ nhà cá nhân — 15,5 so với 12,7 đánh giá/năm."* Hãy thẩm định theo bốn bước, bắt đầu bằng **truy số**.

### Hợp đồng hàm · 15 điểm tự động

`reviews_by_host_type(ds: pd.DataFrame, nguong: int = 5) -> pd.Series`

Nhận bảng chỗ ở có `calculated_host_listings_count` và `number_of_reviews_ltm`. Nhóm theo điều kiện
`calculated_host_listings_count >= nguong` (bool; không thêm cột vào `ds`) và trả Series trung bình `number_of_reviews_ltm`
của từng nhóm: index là các giá trị `False`/`True` có mặt, không làm tròn.
Đối chiếu snapshot: **15,5** (chuyên nghiệp) và **12,7** (cá nhân) đánh giá/năm.

In [15]:
# 12 chỗ ở GIẢ LẬP cho bước truy số; đặt USE_SNAPSHOT = True để dùng listings.csv của snapshot Santiago.
DEMO_DS = pd.DataFrame({
    "id": range(301, 313),
    "calculated_host_listings_count": [1, 7, 2, 12, 5, 1, 3, 9, 1, 6, 2, 4],
    "number_of_reviews_ltm":          [8, 20, 14, 31, 9, 0, 22, 16, 5, 12, 18, 10],
    "review_scores_rating":           [4.9, 4.6, 4.95, 4.7, 4.5, None, 4.85, 4.6, 4.9, 4.55, 4.8, 4.75],
})
USE_SNAPSHOT = False
URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
ds = pd.read_csv(URL) if USE_SNAPSHOT else DEMO_DS.copy()
print("Snapshot Santiago" if USE_SNAPSHOT else "GIẢ LẬP — 12 chỗ ở", "·", len(ds), "dòng")

GIẢ LẬP — 12 chỗ ở · 12 dòng


In [16]:
def reviews_by_host_type(ds: pd.DataFrame, nguong: int = 5) -> pd.Series:
    chuyen = ds["calculated_host_listings_count"].ge(nguong)
    return ds.groupby(chuyen)["number_of_reviews_ltm"].mean()


In [17]:
def check_q5():
    kiem = reviews_by_host_type(DEMO_DS)
    assert isinstance(kiem, pd.Series) and set(kiem.index) == {False, True}
    assert round(kiem[True], 1) == 17.6 and round(kiem[False], 1) == 11.0
    assert "chuyen" not in DEMO_DS.columns
    assert round(reviews_by_host_type(DEMO_DS, 10)[True], 1) == 31.0

public_check("Q5 · ví dụ công khai", check_q5)

[ĐẠT] Q5 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [18]:
preview("Bài 5 — truy số trên dữ liệu đang dùng", lambda: reviews_by_host_type(ds).round(1))

Bài 5 — truy số trên dữ liệu đang dùng
calculated_host_listings_count
False    11.0
True     17.6
Name: number_of_reviews_ltm, dtype: float64


## Bài 6 · Bước 2 (phương pháp) và bước 3 (diễn giải): phán quyết

Chọn **một** phán quyết đúng nhất cho kết luận ở Bài 5:

- **A.** "Giữ nguyên — số đúng thì kết luận đúng."
- **B.** "Sửa cách diễn đạt: số đo là LƯỢNG ĐÁNH GIÁ (biến đại diện cho lượng đặt phòng), không đo 'yêu thích'; muốn nói yêu thích
  phải dùng điểm đánh giá — viết lại thành 'có nhiều đánh giá hơn' và kiểm thêm theo từng nhóm."
- **C.** "Bác bỏ — số liệu sai."

### Câu trả lời · 10 điểm tự động

Gán biến `phan_quyet` là một chữ `"A"`, `"B"` hoặc `"C"`.

In [19]:
phan_quyet = "B"


In [20]:
def check_q6():
    if phan_quyet == "...":
        raise NotImplementedError
    assert phan_quyet in {"A", "B", "C"}, "Chọn một chữ A, B hoặc C"
    print("Định dạng hợp lệ — đáp án do grader chấm.")

public_check("Q6 · định dạng", check_q6)

Định dạng hợp lệ — đáp án do grader chấm.
[ĐẠT] Q6 · định dạng: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


**Chọn B — sửa cách diễn đạt và kiểm thêm.**

1. **Truy số:** với DEMO, nhóm chuyên nghiệp (≥5 listing) có 5 dòng và 88 đánh giá trong 12 tháng, trung bình 88/5 = **17,6**; nhóm cá nhân có 7 dòng và 77 đánh giá, trung bình **11,0**. Các số 15,5 và 12,7 là mốc đối chiếu snapshot do đề cung cấp, không phải kết quả DEMO và chưa được kiểm chứng trên dữ liệu thật.
2. **Kiểm phương pháp:** phép tính nhóm từng listing theo số listing của host rồi lấy trung bình số đánh giá 12 tháng. Đơn vị phân tích là listing, không phải mỗi host một phiếu; host có nhiều listing đóng góp nhiều dòng. Không thể coi số đánh giá là trực tiếp đo mức độ yêu thích.
3. **Sửa diễn giải:** “Trong DEMO, listing thuộc host có ít nhất 5 listing nhận trung bình 17,6 đánh giá trong 12 tháng, so với 11,0 ở nhóm còn lại”. Không kết luận khách yêu thích hơn hoặc chuyên nghiệp hóa gây tăng đánh giá.
4. **Kiểm thêm:** để nghiên cứu mức độ hài lòng, xem điểm đánh giá (`review_scores_rating`), phân phối điểm, số đánh giá làm cơ sở và tỷ lệ thiếu điểm. Kiểm tra theo quận, loại phòng, phân khúc giá, thời gian hoạt động và thời gian mở lịch; phân biệt kết quả theo listing với theo host. Điểm đánh giá chỉ phản ánh khách đã để lại đánh giá và có thể có thiên lệch chọn mẫu; vẫn chưa đủ để suy quan hệ nhân quả.

## Bài tự làm ✅ mở · Thẩm định chéo bản nháp của nhóm — không cộng vào 80 điểm tự động

Chọn một mục trong bản nháp báo cáo của nhóm và: (1) thực hiện quy trình thẩm định bốn bước với
**hai kết luận quan trọng nhất**; (2) kiểm tra từng câu bằng tiêu chí "lời đúng mức" ở Bài 2;
(3) đối chiếu mọi nội dung do AI hỗ trợ viết với mục "AI sai ở đâu" trong `AI_USAGE.md`.
Nếu nhóm có dùng AI nhưng mục này còn trống, hãy bổ sung trước khi nộp.

**Ghi chép thẩm định — minh họa trên nội dung Lab 14**

Chưa có bản nháp báo cáo nhóm hoặc `AI_USAGE.md` trong thư mục Lab để đối chiếu. Hai mục dưới là bài tập thẩm định dựa trên dữ liệu đã cho, chưa phải xác nhận đã thẩm định chéo báo cáo của nhóm.

**Kết luận 1: “Giá gộp giảm nên giá ở cả hai phân khúc đều giảm.”**

1. **Truy số:** trung bình gộp giảm từ 45,0 xuống 41,8, nhưng nhóm cao tăng 60 → 66, nhóm rẻ tăng 30 → 33.
2. **Kiểm phương pháp:** trung bình gộp có trọng số theo số chỗ ở; tỷ trọng nhóm rẻ tăng 50% → 73,33%.
3. **Sửa lời:** “Trong bảng minh họa, giá trung bình gộp giảm 7,11% do cơ cấu dịch chuyển về nhóm rẻ, trong khi giá trung bình của cả hai nhóm đều tăng 10%”. Không gọi đây là kết quả thị trường Santiago.
4. **Kiểm thêm:** giữ trọng số mỗi nhóm 50% cho T6 thì giá gộp là 49,5, cho thấy riêng thay đổi giá trong nhóm làm tăng 10%; cần kiểm tra cơ cấu tương tự khi phân tích snapshot thật.

**Kết luận 2: “Chủ nhà chuyên nghiệp được khách yêu thích hơn.”**

1. **Truy số:** DEMO cho trung bình số đánh giá 17,6 so với 11,0 trên mỗi listing.
2. **Kiểm phương pháp:** số lượng đánh giá không phải điểm hài lòng; mẫu gồm nhiều listing của cùng host có thể không độc lập.
3. **Sửa lời:** “Trong DEMO, listing thuộc nhóm host có ít nhất 5 listing có số đánh giá 12 tháng trung bình cao hơn”. Bỏ kết luận về yêu thích và nguyên nhân.
4. **Kiểm thêm:** kiểm tra điểm đánh giá theo nhóm, số quan sát, dữ liệu thiếu, quận và loại phòng; không đồng nhất DEMO với snapshot.

**Đối chiếu nội dung do AI hỗ trợ:** các phép tính minh họa đã được kiểm tra bằng Python chuẩn. Chưa có `AI_USAGE.md` của nhóm để kiểm tra mục “AI sai ở đâu”; không tự ghi nhận lỗi AI hoặc hoạt động nhóm chưa xảy ra. Cần thay hai ví dụ trên bằng hai kết luận thực tế và dẫn số liệu của báo cáo khi có bản nháp.

## Khai báo sử dụng AI

- **Công cụ đã dùng:** OpenAI Codex.
- **Bài đã dùng AI và yêu cầu chính:** hai bài tự làm và các phần diễn giải theo yêu cầu “Cho tôi khung code và giải thích chi tiết”.
- **Kiểm chứng:** chạy ba public check định dạng cho Bài 1, 2, 6 bằng Python chuẩn; kiểm tra cú pháp các cell. Tính độc lập bằng Python chuẩn: (100×60 + 100×30)/200 = 45; (80×66 + 220×33)/300 = 41,8; tỷ trọng rẻ 50% → 73,33%; số đánh giá trung bình 88/5 = 17,6 và 77/7 = 11.
- **Giới hạn:** public check định dạng không xác nhận đúng/sai nội dung đáp án. Notebook đã lưu 6 public checks báo ĐẠT, gồm các bài dùng pandas; chưa chạy lại từ đầu trong phiên cập nhật này; chưa kiểm chứng snapshot hoặc thẩm định bản nháp nhóm.


---

## 🧭 Hỗ trợ bài tập lớn (~30 phút — làm theo nhóm, không tính điểm lab)

Hạn nộp: **23:59 Chủ nhật tuần này** (ngày cụ thể: Canvas Portal). Hãy kiểm tra theo danh sách
trong slide Bài 15; dưới đây là bản rút gọn để nhóm tự rà soát:

1. ☐ `run_pipeline.py`/Makefile: **một lệnh** chạy từ dữ liệu gốc đến mọi kết quả; thành phố và
   mốc chụp nằm trong `configs/`, không ghi cứng trong mã.
2. ☐ Bộ nhớ đệm LLM đầy đủ; `--skip-llm` chạy **không cần khoá API**; khoá không xuất hiện
   trong kho mã nguồn, kể cả lịch sử commit (kiểm tra bằng `grep`).
3. ☐ `data/samples/` có ít nhất 100 nhãn tay; `reports/` có PDF và `qa_report.csv`;
   `figures/` sinh tự động.
4. ☐ `AI_USAGE.md` cập nhật lần cuối — mục "AI sai ở đâu" có ví dụ thật.
5. ☐ **Bài kiểm tra máy sạch** đã được lên lịch: một thành viên chạy `git clone` vào thư mục mới và
   làm theo README trước thứ Bảy; gắn thẻ `final` và ghi mã commit vào báo cáo.

> Giảng viên thực hành trao đổi với từng nhóm về các mục chưa hoàn tất và người phụ trách trước
> thứ Bảy; các nhóm có nguy cơ trễ cần được báo lại cho giảng viên sau buổi học.

## Tóm tắt bài thực hành

| Bạn đã làm | Dùng cho |
|---|---|
| Kim tự tháp: kết luận trước, phương pháp sau | mọi mục của báo cáo nộp tuần này |
| Năm cặp số liệu–kết luận: đúng mức hay quá mức | tự kiểm tra từng câu trước khi nộp |
| Tự tính nghịch lý Simpson: cơ cấu nhóm làm số gộp đổi chiều | so sánh hai mốc chụp trong báo cáo |
| Thẩm định "yêu thích" và "có nhiều đánh giá hơn" | chuẩn bị cho các câu hỏi tương tự trong vấn đáp |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

Tuần sau là buổi **vấn đáp**. Hãy chuẩn bị theo slide Bài 15 và kế hoạch ôn tập trong đó,
đồng thời hoàn thành việc nộp bài đúng hạn.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** Nguyễn Tiến Long An / 25022763 / DSE2049_3

1. Giữ tên file **`lab-14.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_14.ipynb`).
2. Hoàn thành các câu trả lời, hàm TODO, lập luận và khai báo AI.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`.
4. Tạo tag `submit/lab-14/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-14` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [21]:
show_public_summary()

PUBLIC CHECKS — không phải điểm chính thức
Q1 · định dạng: ĐẠT
Q2 · định dạng: ĐẠT
Q3 · ví dụ công khai: ĐẠT
Q4 · ví dụ công khai: ĐẠT
Q5 · ví dụ công khai: ĐẠT
Q6 · định dạng: ĐẠT
Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.
